In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

### load data

In [2]:
aapl = pd.read_csv('AAPL_2023-11_2024-11.csv')
aapl

,timestamp,volume,open,close,high,low,Volatility,depth,symbol,bid_px_00,...,ask_px_05,ask_sz_05,ask_px_06,ask_sz_06,ask_px_07,ask_sz_07,ask_px_08,ask_sz_08,ask_px_09,ask_sz_09
0,2023-11-09 13:30:00+00:00,425,182.4900,182.4900,182.49,182.4900,NaN,0,AAPL,182.35,...,182.67,1,182.69,90,182.72,15,182.73,1,182.78,2
1,2023-11-09 13:32:00+00:00,1821,182.3500,182.3000,182.35,182.3000,NaN,9,AAPL,182.35,...,182.65,103,182.67,1,182.69,90,182.72,15,182.73,3
2,2023-11-09 13:33:00+00:00,4082,182.2602,182.2500,182.30,182.2500,NaN,1,AAPL,182.26,...,182.54,50,182.59,4,182.60,107,182.65,103,182.67,1
3,2023-11-09 13:34:00+00:00,2641,182.2300,182.2900,182.29,182.2300,NaN,7,AAPL,182.21,...,182.59,4,182.60,107,182.65,103,182.67,1,182.69,90
4,2023-11-09 13:35:00+00:00,5440,182.3000,182.3100,182.31,182.2900,NaN,2,AAPL,182.28,...,182.54,50,182.59,4,182.60,107,182.65,103,182.67,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103392,2024-11-08 19:56:00+00:00,26591,227.4535,227.4500,227.49,227.4301,0.072883,9,AAPL,228.09,...,228.16,245,228.17,668,228.18,313,228.19,1459,228.20,397
103393,2024-11-08 19:57:00+00:00,23227,227.4400,227.4450,227.45,227.4200,0.073030,9,AAPL,227.92,...,228.00,253,228.01,226,228.02,283,228.03,467,228.04,454
103394,2024-11-08 19:58:00+00:00,23071,227.4450,227.4600,227.48,227.4300,0.073246,9,AAPL,227.90,...,227.96,335,227.97,157,227.98,352,227.99,173,228.00,446
103395,2024-11-08 19:59:00+00:00,25473,227.4600,227.4400,227.47,227.4400,0.074191,9,AAPL,227.84,...,227.90,131,227.91,140,227.92,516,227.93,152,227.94,298


In [3]:
data = aapl[aapl.timestamp.str.contains('2024-11-08')].set_index('timestamp',drop=True)
data.index = pd.to_datetime(data.index)
data

,volume,open,close,high,low,Volatility,depth,symbol,bid_px_00,bid_sz_00,...,ask_px_05,ask_sz_05,ask_px_06,ask_sz_06,ask_px_07,ask_sz_07,ask_px_08,ask_sz_08,ask_px_09,ask_sz_09
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-11-08 14:30:00+00:00,770880,227.1700,227.6400,227.7600,227.1400,0.167021,5,AAPL,227.72,483,...,229.80,1000,229.90,10,230.00,28,230.78,6,230.80,8
2024-11-08 14:31:00+00:00,237773,227.6400,228.1401,228.1700,227.4700,0.179580,4,AAPL,227.72,240,...,229.70,111,229.80,1000,229.90,10,230.00,28,230.78,6
2024-11-08 14:32:00+00:00,221508,228.1700,228.4501,228.4594,228.0700,0.241811,4,AAPL,227.72,240,...,229.70,111,229.80,1000,229.90,10,230.00,28,230.78,6
2024-11-08 14:33:00+00:00,192968,228.4600,228.3800,228.6600,228.3700,0.318099,4,AAPL,227.72,240,...,229.70,111,229.80,1000,229.90,10,230.00,28,230.78,6
2024-11-08 14:34:00+00:00,155137,228.3700,228.1250,228.3800,228.0000,0.339953,7,AAPL,227.75,441,...,229.80,1000,229.90,10,230.00,128,230.78,6,230.80,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-08 19:56:00+00:00,26591,227.4535,227.4500,227.4900,227.4301,0.072883,9,AAPL,228.09,302,...,228.16,245,228.17,668,228.18,313,228.19,1459,228.20,397
2024-11-08 19:57:00+00:00,23227,227.4400,227.4450,227.4500,227.4200,0.073030,9,AAPL,227.92,181,...,228.00,253,228.01,226,228.02,283,228.03,467,228.04,454
2024-11-08 19:58:00+00:00,23071,227.4450,227.4600,227.4800,227.4300,0.073246,9,AAPL,227.90,282,...,227.96,335,227.97,157,227.98,352,227.99,173,228.00,446


In [4]:
def triple_barrier(prices, events, pt_sl):
    """   
    :param prices: (pd.Series) Close prices
    :param events:  (pd.DataFrame): with 't1' (max holding period end time) and 'side' (trade direction).
    :param holding_period: (int) Number of minutes to add for vertical barrier
    """
    
    out = events[['t1']].copy()
    pt, sl = pt_sl
    profit_taking = pt * events['vol']
    stop_loss =  -sl * events['vol']

    for loc, t1 in events['t1'].fillna(prices.index[-1]).items():
        # Get the price series between the entry time and the barrier time
        price_path = prices[loc:t1]
        
        # Calculate the thresholds
        initial_price = prices[loc]

        # get cumreturn
        cum_ret = (price_path / initial_price - 1) * events.at[loc, 'side']

        # fist target time to pt
        out.at[loc, 'sl'] = cum_ret[cum_ret < stop_loss[loc]].index.min()
        # first target time to sl
        out.at[loc, 'pt'] = cum_ret[cum_ret > profit_taking[loc]].index.min() 

    for ind in out.index:
        out.at[ind, 'exit'] = out.loc[ind, ['t1','sl','pt']].dropna().min()
    
    return out

In [5]:
def meta_labeling(prices, vol, pt_sl, holding_period, signal):
    """
    :param prices: (pd.Series) Close prices
    :param pt_sl: (list) [profit-taking threshold, stop-loss threshold].
    :param holding_period: (int) Maximum holding period in minutes.
    """

    # Step 1: Define events
    events = pd.DataFrame(index=prices.index)
    events['side'] = signal
    events['vol'] = vol
    
    # add vertical barrier
    max_t1 = prices.index[-1]
    events['t1'] = prices.index + pd.Timedelta(minutes=holding_period)
    events['t1'] = np.minimum(events['t1'], max_t1)
    
    # Step 2: Apply triple-barrier method
    outcomes = triple_barrier(prices, events, pt_sl)

    # Step 3: Create meta-labels
    meta_labels = pd.DataFrame(index=outcomes.index)
    meta_labels['exit'] = outcomes['exit']
    meta_labels['price'] = prices
    meta_labels['ret'] = 0
    meta_labels.loc[events.index,'ret'] = (prices.loc[outcomes['exit'].array].array - prices.loc[outcomes.index])/ prices.loc[outcomes.index] * events['side']
    
    return meta_labels

### generate feature

In [6]:
from ta import momentum, trend, volatility, volume

In [7]:
def get_ta(df_):
    """
    :param df_: (DataFrame) contain [open,high,low,close,volume] in columns.
    """
    df = df_.copy()   

    df['m_rsi'] = momentum.rsi(df.close)
    df['m_roc'] = momentum.roc(df.close)
    df['m_wr']  = momentum.williams_r(df.high, df.low, df.close)
    df['vm_fi'] = volume.force_index(df.close, df.volume)
    df['vm_eom'] = volume.ease_of_movement(df.high, df.low, df.volume)
    df['vl_bbp'] = volatility.bollinger_pband(df.close)
    df['vl_atr'] = volatility.average_true_range(df.high, df.low, df.close)
    df['t_macdd']  = trend.MACD(df.close).macd_diff()
    df['t_trix'] = trend.trix(df.close)
    df['t_cci'] = trend.cci(df.high, df.low, df.close)
            
    return df 

In [8]:
fea_df = get_ta(data[['volume','open','close','high','low']])

In [9]:
fea_df

,volume,open,close,high,low,m_rsi,m_roc,m_wr,vm_fi,vm_eom,vl_bbp,vl_atr,t_macdd,t_trix,t_cci
timestamp,,,,,,,,,,,,,,,
2024-11-08 14:30:00+00:00,770880,227.1700,227.6400,227.7600,227.1400,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
2024-11-08 14:31:00+00:00,237773,227.6400,228.1401,228.1700,227.4700,NaN,NaN,NaN,NaN,108.927422,NaN,0.000000,NaN,NaN,NaN
2024-11-08 14:32:00+00:00,221508,228.1700,228.4501,228.4594,228.0700,NaN,NaN,NaN,NaN,78.176039,NaN,0.000000,NaN,NaN,NaN
2024-11-08 14:33:00+00:00,192968,228.4600,228.3800,228.6600,228.3700,NaN,NaN,NaN,NaN,37.616081,NaN,0.000000,NaN,NaN,NaN
2024-11-08 14:34:00+00:00,155137,228.3700,228.1250,228.3800,228.0000,NaN,NaN,NaN,NaN,-79.607057,NaN,0.000000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-08 19:56:00+00:00,26591,227.4535,227.4500,227.4900,227.4301,44.296113,-0.092243,-76.666667,-344.007887,5.732974,0.260787,0.065315,-0.016935,0.000164,-64.940951
2024-11-08 19:57:00+00:00,23227,227.4400,227.4450,227.4500,227.4200,43.781674,-0.055807,-76.785714,-311.454617,-3.235459,0.247178,0.062792,-0.015084,-0.000088,-83.561228
2024-11-08 19:58:00+00:00,23071,227.4450,227.4600,227.4800,227.4300,45.814757,-0.059316,-71.428571,-217.523243,4.334446,0.299002,0.061879,-0.012156,-0.000301,-63.951226


In [10]:
features = fea_df.dropna(how='any',axis=0)

In [11]:
features

,volume,open,close,high,low,m_rsi,m_roc,m_wr,vm_fi,vm_eom,vl_bbp,vl_atr,t_macdd,t_trix,t_cci
timestamp,,,,,,,,,,,,,,,
2024-11-08 15:13:00+00:00,85601,227.9600,227.8600,228.0900,227.8549,47.762817,-0.008777,-40.350877,435.512668,6.165810,0.389593,0.208562,0.008074,-0.001040,23.643232
2024-11-08 15:14:00+00:00,62779,227.8547,227.7676,227.9799,227.7600,44.225373,-0.056035,-56.561404,-455.386228,-35.903328,0.195835,0.209372,-0.003243,-0.001011,-51.695547
2024-11-08 15:15:00+00:00,85731,227.7600,227.7563,227.8800,227.6200,43.798158,-0.080591,-71.000000,-528.725381,-36.377740,0.184462,0.212988,-0.010842,-0.001092,-115.030190
2024-11-08 15:16:00+00:00,33287,227.7800,227.8503,227.8699,227.7550,48.274411,-0.004257,-51.000000,-6.196327,21.556479,0.446633,0.205982,-0.008969,-0.001161,-43.722270
2024-11-08 15:17:00+00:00,40395,227.8600,227.9100,227.9500,227.8200,50.946584,0.061466,-38.297872,339.200506,23.348187,0.607031,0.200555,-0.003425,-0.001165,18.294747
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-08 19:56:00+00:00,26591,227.4535,227.4500,227.4900,227.4301,44.296113,-0.092243,-76.666667,-344.007887,5.732974,0.260787,0.065315,-0.016935,0.000164,-64.940951
2024-11-08 19:57:00+00:00,23227,227.4400,227.4450,227.4500,227.4200,43.781674,-0.055807,-76.785714,-311.454617,-3.235459,0.247178,0.062792,-0.015084,-0.000088,-83.561228
2024-11-08 19:58:00+00:00,23071,227.4450,227.4600,227.4800,227.4300,45.814757,-0.059316,-71.428571,-217.523243,4.334446,0.299002,0.061879,-0.012156,-0.000301,-63.951226


In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [13]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
fea_norm = pd.DataFrame(scaler.fit_transform(features),index=features.index)

In [14]:
close = data.close
close_ret = close.pct_change().fillna(0)
y = np.sign(close_ret.reindex(fea_norm.index))
y

timestamp
2024-11-08 15:13:00+00:00   -1.0
2024-11-08 15:14:00+00:00   -1.0
2024-11-08 15:15:00+00:00   -1.0
2024-11-08 15:16:00+00:00    1.0
2024-11-08 15:17:00+00:00    1.0
                            ... 
2024-11-08 19:56:00+00:00   -1.0
2024-11-08 19:57:00+00:00   -1.0
2024-11-08 19:58:00+00:00    1.0
2024-11-08 19:59:00+00:00   -1.0
2024-11-08 20:00:00+00:00    1.0
Name: close, Length: 288, dtype: float64

In [15]:
# ignore the zero part
where = (y!=0)

In [16]:
y[where]

timestamp
2024-11-08 15:13:00+00:00   -1.0
2024-11-08 15:14:00+00:00   -1.0
2024-11-08 15:15:00+00:00   -1.0
2024-11-08 15:16:00+00:00    1.0
2024-11-08 15:17:00+00:00    1.0
                            ... 
2024-11-08 19:56:00+00:00   -1.0
2024-11-08 19:57:00+00:00   -1.0
2024-11-08 19:58:00+00:00    1.0
2024-11-08 19:59:00+00:00   -1.0
2024-11-08 20:00:00+00:00    1.0
Name: close, Length: 281, dtype: float64

In [35]:
X_train, X_test = fea_norm[where].iloc[:200,:],fea_norm[where].iloc[200:,:]
y_train, y_test = y[where].iloc[:200],y[where].iloc[200:]

In [36]:
# Train the rf model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)
print('train data:')
print(classification_report(y_train, y_train_pred))
print(confusion_matrix(y_train, y_train_pred))
print('test data:')
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))

train data:
              precision    recall  f1-score   support

        -1.0       1.00      1.00      1.00       101
         1.0       1.00      1.00      1.00        99

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

[[101   0]
 [  0  99]]
test data:
              precision    recall  f1-score   support

        -1.0       0.56      0.97      0.71        35
         1.0       0.95      0.41      0.58        46

    accuracy                           0.65        81
   macro avg       0.75      0.69      0.64        81
weighted avg       0.78      0.65      0.63        81

[[34  1]
 [27 19]]


In [37]:
y_score = rf_model.predict_proba(X_train)[:,1]

In [38]:
y_score

array([0.22, 0.03, 0.1 , 0.91, 0.95, 0.72, 0.87, 0.93, 0.27, 0.05, 0.03,
       0.09, 0.7 , 0.18, 0.11, 0.1 , 0.72, 0.85, 0.15, 0.07, 0.71, 0.13,
       0.78, 0.14, 0.08, 0.12, 0.85, 0.08, 0.99, 0.18, 0.78, 0.96, 0.17,
       0.01, 0.68, 0.98, 0.02, 0.07, 0.86, 0.96, 0.19, 0.17, 0.79, 0.94,
       0.1 , 0.02, 0.08, 0.9 , 0.26, 0.82, 0.94, 0.97, 0.1 , 0.2 , 0.76,
       0.78, 0.27, 0.82, 0.09, 0.91, 0.85, 0.11, 0.86, 0.92, 0.96, 0.95,
       0.19, 0.96, 0.93, 0.3 , 0.76, 0.18, 0.08, 0.02, 0.09, 0.94, 0.28,
       0.98, 0.98, 0.82, 0.98, 1.  , 0.2 , 0.85, 1.  , 0.93, 0.18, 0.11,
       0.86, 0.95, 0.98, 0.08, 0.12, 0.82, 0.77, 0.97, 0.22, 0.75, 0.86,
       0.23, 0.86, 0.22, 0.09, 0.12, 0.11, 0.75, 0.13, 0.78, 0.17, 0.05,
       0.06, 0.06, 0.71, 0.25, 0.81, 0.31, 0.04, 0.09, 0.85, 0.96, 0.77,
       0.16, 0.75, 0.13, 0.1 , 0.85, 0.36, 0.01, 0.02, 0.01, 0.84, 0.96,
       0.22, 0.16, 0.73, 1.  , 0.99, 0.91, 0.32, 0.98, 0.91, 0.34, 0.16,
       0.08, 0.06, 0.8 , 0.11, 0.06, 0.14, 0.12, 0.

In [43]:
# adjust threshold to imporve recall
thres = 0.2

In [44]:
prediction_high_recall = 2*(y_score > thres).astype(int)-1

In [45]:
prediction_high_recall

array([ 1, -1, -1,  1,  1,  1,  1,  1,  1, -1, -1, -1,  1, -1, -1, -1,  1,
        1, -1, -1,  1, -1,  1, -1, -1, -1,  1, -1,  1, -1,  1,  1, -1, -1,
        1,  1, -1, -1,  1,  1, -1, -1,  1,  1, -1, -1, -1,  1,  1,  1,  1,
        1, -1, -1,  1,  1,  1,  1, -1,  1,  1, -1,  1,  1,  1,  1, -1,  1,
        1,  1,  1, -1, -1, -1, -1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,
        1, -1, -1,  1,  1,  1, -1, -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
       -1, -1, -1,  1, -1,  1, -1, -1, -1, -1,  1,  1,  1,  1, -1, -1,  1,
        1,  1, -1,  1, -1, -1,  1,  1, -1, -1, -1,  1,  1,  1, -1,  1,  1,
        1,  1,  1,  1,  1,  1, -1, -1, -1,  1, -1, -1, -1, -1, -1,  1,  1,
        1,  1, -1,  1,  1, -1, -1, -1, -1, -1, -1,  1,  1, -1,  1,  1, -1,
        1,  1, -1, -1, -1, -1,  1, -1,  1,  1, -1, -1, -1, -1,  1,  1,  1,
        1,  1,  1, -1,  1,  1,  1, -1,  1,  1,  1,  1,  1])

In [46]:
print(classification_report(y_train, prediction_high_recall))
print(confusion_matrix(y_train, prediction_high_recall))

              precision    recall  f1-score   support

        -1.0       1.00      0.83      0.91       101
         1.0       0.85      1.00      0.92        99

    accuracy                           0.92       200
   macro avg       0.93      0.92      0.91       200
weighted avg       0.93      0.92      0.91       200

[[84 17]
 [ 0 99]]


In [25]:
signals = pd.Series(0, index=y.index)[:y_train.index[-1]]
signals[y_train.index] = prediction_high_recall
signals

timestamp
2024-11-08 15:13:00+00:00    1
2024-11-08 15:14:00+00:00   -1
2024-11-08 15:15:00+00:00   -1
2024-11-08 15:16:00+00:00    1
2024-11-08 15:17:00+00:00    1
                            ..
2024-11-08 19:02:00+00:00    1
2024-11-08 19:03:00+00:00   -1
2024-11-08 19:04:00+00:00    1
2024-11-08 19:05:00+00:00    1
2024-11-08 19:06:00+00:00    1
Length: 234, dtype: int64

In [26]:
# Profit-taking: 1%, Stop-loss: 0.5%
pt_sl = [0.001, 0.001] 
holding_period = 10 

In [27]:
vol = data.Volatility
meta_label = meta_labeling(close[signals.index], vol[signals.index], pt_sl, holding_period,signals)

In [28]:
meta_label

,exit,price,ret
timestamp,,,
2024-11-08 15:13:00+00:00,2024-11-08 15:14:00+00:00,227.8600,-0.000406
2024-11-08 15:14:00+00:00,2024-11-08 15:16:00+00:00,227.7676,-0.000363
2024-11-08 15:15:00+00:00,2024-11-08 15:16:00+00:00,227.7563,-0.000413
2024-11-08 15:16:00+00:00,2024-11-08 15:17:00+00:00,227.8503,0.000262
2024-11-08 15:17:00+00:00,2024-11-08 15:18:00+00:00,227.9100,0.000110
...,...,...,...
2024-11-08 19:02:00+00:00,2024-11-08 19:03:00+00:00,227.4400,-0.000198
2024-11-08 19:03:00+00:00,2024-11-08 19:05:00+00:00,227.3950,-0.000308
2024-11-08 19:04:00+00:00,2024-11-08 19:05:00+00:00,227.4150,0.000220


In [29]:
meta_label['meta_label'] = np.sign(meta_label.ret)

In [30]:
meta_label.meta_label.value_counts()

meta_label
 1.0    119
-1.0    110
 0.0      5
Name: count, dtype: int64

In [31]:
new_X = pd.concat([fea_norm,meta_label.meta_label],axis=1).loc[y_train.index]

In [32]:
new_X.columns = new_X.columns.astype(str)

In [33]:
new_X

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,meta_label
timestamp,,,,,,,,,,,,,,,,
2024-11-08 15:13:00+00:00,0.371694,2.496596,2.191501,2.815521,2.297701,-0.192043,-0.004196,0.408017,0.267101,0.403948,-0.298647,2.763049,0.346786,-0.078968,0.271002,-1.0
2024-11-08 15:14:00+00:00,0.116832,2.151238,1.885893,2.445426,1.987258,-0.517668,-0.447840,-0.111510,-0.240308,-2.108867,-0.905248,2.785108,-0.135847,-0.073320,-0.441122,-1.0
2024-11-08 15:15:00+00:00,0.373146,1.840645,1.848519,2.109618,1.529281,-0.556994,-0.678372,-0.574249,-0.282078,-2.137204,-0.940854,2.883610,-0.459955,-0.089085,-1.039780,-1.0
2024-11-08 15:16:00+00:00,-0.212515,1.906240,2.159419,2.075667,1.970902,-0.144951,0.038232,0.066726,0.015527,1.323242,-0.120074,2.692769,-0.380058,-0.102434,-0.365757,1.0
2024-11-08 15:17:00+00:00,-0.133138,2.168620,2.356874,2.344919,2.183534,0.101025,0.655214,0.473813,0.212247,1.430262,0.382086,2.544939,-0.143611,-0.103196,0.220446,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-08 19:02:00+00:00,-0.241673,1.069902,0.802374,1.000341,0.940453,1.063653,1.151344,0.462965,1.365560,-1.025085,0.803320,-0.495929,0.630957,0.887702,1.017252,-1.0
2024-11-08 19:03:00+00:00,-0.238681,0.758982,0.653539,0.670920,0.678752,0.630848,0.925273,0.135194,1.059905,-1.544853,0.454834,-0.489929,0.306464,0.908685,0.460029,-1.0
2024-11-08 19:04:00+00:00,-0.209266,0.627135,0.719688,0.630582,0.744177,0.771754,0.987235,0.280870,0.965853,0.078353,0.575938,-0.546616,0.116892,0.918360,0.489794,1.0


In [34]:
# Train the meta-model
meta_model = RandomForestClassifier()
meta_model.fit(new_X, y_train)

# Evaluate
new_y_pred = meta_model.predict(new_X)
print('meta model: ')
print(classification_report(y_train, new_y_pred))
print(confusion_matrix(y_train, new_y_pred))

meta model: 
              precision    recall  f1-score   support

        -1.0       1.00      1.00      1.00       112
         1.0       1.00      1.00      1.00       118

    accuracy                           1.00       230
   macro avg       1.00      1.00      1.00       230
weighted avg       1.00      1.00      1.00       230

[[112   0]
 [  0 118]]
